In [ ]:
import numpy as np
import pandas as pd

from imblearn.combine import SMOTEENN
from imblearn.over_sampling import BorderlineSMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from models.MLPipeline import *
from utils import ASSETS_DIR

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.impute import KNNImputer
from sklearn.neural_network import MLPClassifier
from sklearn import config_context

In [ ]:
df = pd.read_parquet(ASSETS_DIR / "final_df.parquet")

In [ ]:


# 1. Separiamo la matrice delle Feature (X) dal Target (y)
X = df.drop(columns=['TARGET'])
X.drop(columns=['LBDEVAL'], inplace=True, errors='ignore')
y = df['TARGET']

# 2. TRAIN-TEST SPLIT (80% Train, 20% Test)
# stratify=y è fondamentale per mantenere le stesse percentuali di malati nei due set
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)

print(f"Buchi (NaN) iniziali in X_train: {X_train.isna().sum().sum()}")

# 3. CONFIGURAZIONE DEL KNN IMPUTER
# weights='distance' dà più importanza ai vicini più vicini (più simili)
imputer = KNNImputer(n_neighbors=7, weights='distance')

# 4. ADDESTRAMENTO E TRASFORMAZIONE
# Il modello "impara" le distribuzioni SOLO da X_train
X_train_imp = pd.DataFrame(imputer.fit_transform(X_train), columns=X_train.columns)

# Il modello applica quanto imparato su X_test (senza barare)
X_test_imp = pd.DataFrame(imputer.transform(X_test), columns=X_test.columns)

# 5. ARROTONDAMENTO PER DATI CLINICI DISCRETI
# Riportiamo le medie del KNN a numeri interi (es. 0.66 diventa 1.0)
X_train_imp = X_train_imp.round()
X_test_imp = X_test_imp.round()

print(f"Buchi (NaN) finali in X_train: {X_train_imp.isna().sum().sum()}")
print(f"Buchi (NaN) finali in X_test: {X_test_imp.isna().sum().sum()}")


In [ ]:
from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import numpy as np

# 1. MIN-MAX SCALING (Deve avvenire SEMPRE prima della PCA)
scaler = MinMaxScaler()

# Il modello impara i minimi e massimi SOLO dal Train Set
X_train_scaled = scaler.fit_transform(X_train_imp)
# Scala il Test Set basandosi su quanto imparato (No Leakage)
X_test_scaled = scaler.transform(X_test_imp)

# 2. APPLICAZIONE DELLA PCA
# Invece di scegliere un numero fisso di componenti a caso, chiediamo a scikit-learn
# di tenere un numero di componenti sufficiente a spiegare il 90% della varianza totale.
pca = PCA(n_components=0.90, random_state=42)

X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

print(f"Dimensioni originali: {X_train_scaled.shape[1]} features")
print(f"Dimensioni dopo PCA: {X_train_pca.shape[1]} componenti principali (spiegano il 90% della varianza)")

# 3. VISUALIZZAZIONE DELLA VARIANZA SPIEGATA
plt.figure(figsize=(8, 5))
plt.plot(np.cumsum(pca.explained_variance_ratio_), marker='o', linestyle='--')
plt.title("Varianza Cumulativa Spiegata dalle Componenti Principali")
plt.xlabel("Numero di Componenti")
plt.ylabel("Varianza Spiegata")
plt.grid(True)
plt.show()

In [ ]:

mlp = MLPClassifier((200, 250))


smote_enn = SMOTEENN(random_state=42)
smote_nc = BorderlineSMOTE(random_state=42, k_neighbors=7)
pipeline_xgb = ImbPipeline(steps=[
    ('smoteenn', smote_nc),
    ('model', mlp)
])

y_true_bin, y_proba = train_model_evaluate(X_train_imp, y_train, pipeline_xgb)


In [ ]:
generate_predictions_and_cm(X_train_imp, y_train, pipeline_xgb)